In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/hdac_data'      # Drive에서 폴더 이름의 공백을 지운 뒤
print([repr(f) for f in os.listdir(DATA_DIR)])     # 파일 이름 확인

!pip -q install pyreadstat
import pyreadstat
PATH = f'{DATA_DIR}/HN24_ALL.sav'

# 변수 이름과 라벨만 먼저 읽기 (자료 전체를 읽지 않아 빠름)
_, meta = pyreadstat.read_sav(PATH, metadataonly=True)
labels = meta.column_names_to_labels

def find_vars(*keywords):
    """이름이나 라벨에 키워드가 들어간 변수를 모두 보여 준다."""
    for name, label in labels.items():
        if any(k in f'{name} {label or ""}' for k in keywords):
            print(f'{name:15s} {label}')

find_vars('수면', '잠', '취침', '기상', '일어')     # 수면 문항
find_vars('결혼')                  # 결혼상태
find_vars('경제활동', '취업')       # 경제활동
find_vars('근로', '근무', '일한')   # 근로시간, 근로시간대
find_vars('우울', 'PHQ')           # 우울
find_vars('불안', 'GAD')           # 불안
find_vars('의사진단')              # 만성질환

Mounted at /content/drive
["'hn24_all.sas7bdat'", "'HN24_ALL.sav'", "'09_data_check.ipynb'", "'Untitled0.ipynb'"]
BP16_11         주중 잠자리에 든 시각(시간)
BP16_12         주중 잠자리에 든 시각(분)
BP16_13         주중 일어난 시각(시간)
BP16_14         주중 일어난 시각(분)
BP16_21         주말 잠자리에 든 시각(시간)
BP16_22         주말 잠자리에 든 시각(분)
BP16_23         주말 일어난 시각(시간)
BP16_24         주말 일어난 시각(분)
HE_Ubld         요잠혈
BM1_8           칫솔질 시기: 잠자기 전
marri_1         결혼여부
marri_2         결혼상태
occp            직업재분류 및 실업/비경제활동인구상태 코드
EC1_1           경제활동 상태
EC_stt_2        종사상지위: 임금근로자 상세
EC_wht_23       주당 평균 근로시간
EC_wht_5        주/변형근로시간
EC_wht_6        변형근로시간:기타상세
LQ4_15          활동제한 사유: 우울, 불안, 정서상 문제
LQ_5EQL         EuroQoL: 불안, 우울
BP_PHQ_1        PHQ-9 문항1
BP_PHQ_2        PHQ-9 문항2
BP_PHQ_3        PHQ-9 문항3
BP_PHQ_4        PHQ-9 문항4
BP_PHQ_5        PHQ-9 문항5
BP_PHQ_6        PHQ-9 문항6
BP_PHQ_7        PHQ-9 문항7
BP_PHQ_8        PHQ-9 문항8
BP_PHQ_9        PHQ-9 문항9
mh_PHQ_S        PHQ-9 점수
BP5             2주이상 연속 우울감 여부
LQ4_15  

In [2]:
import numpy as np
import pandas as pd

df, meta = pyreadstat.read_sav(PATH, usecols=['age', 'sex', 'BP1', 'mh_stress'])   # 4단계에서 공변량을 더함

df['stress'] = np.select([df['BP1'].isin([1, 2]), df['BP1'].isin([3, 4])], [1, 0], default=np.nan)

print(pd.crosstab(df['BP1'], df['stress'], dropna=False))       # 원 코드가 맞게 묶였는지
print(pd.crosstab(df['stress'], df['mh_stress'], dropna=False)) # mh_stress와 같은지

stress   0.0   1.0  NaN
BP1                    
1.0        0   297    0
2.0        0  1216    0
3.0     3409     0    0
4.0     1286     0    0
8.0        0     0  561
9.0        0     0  219
NaN        0     0    9
mh_stress   0.0   1.0  NaN
stress                    
0.0        4695     0    0
1.0           0  1513    0
NaN           0     0  789


In [4]:
# 1단계 출력 및 변수 목록에서 확인한 실제 변수명으로 매핑
WD = dict(
    bed_h='BP16_11', bed_m='BP16_12', wake_h='BP16_13', wake_m='BP16_14'
)  # 주중 (잠자리에 든 시각 시간/분, 일어난 시각 시간/분)
WE = dict(
    bed_h='BP16_21', bed_m='BP16_22', wake_h='BP16_23', wake_m='BP16_24'
)  # 주말 (잠자리에 든 시각 시간/분, 일어난 시각 시간/분)
SPECIAL = [88, 99]  # 결측치나 해당 없음 처리할 특수문항 코드 (지침에 맞게 설정)                                                          # 확인 필요

def clean(col):
    return df[col].where(~df[col].isin(SPECIAL))

def sleep_hours(v):
    bed  = clean(v['bed_h'])  * 60 + clean(v['bed_m'])
    wake = clean(v['wake_h']) * 60 + clean(v['wake_m'])
    return ((wake - bed) % (24 * 60)) / 60      # 자정을 넘겨 잠들어도 맞게 계산

df['sleep_wd'] = sleep_hours(WD)
df['sleep_we'] = sleep_hours(WE)
df['sleep'] = (df['sleep_wd'] * 5 + df['sleep_we'] * 2) / 7

KeyError: 'BP16_11'

In [ ]:
print(df[['sleep']].describe())                          # 최솟값·최댓값이 상식적인지
print((df['sleep'] < 3).sum(), (df['sleep'] > 16).sum())  # 이상값 후보 수

In [ ]:
# 1. 이상값(3시간 미만 또는 16시간 초과)을 제외한 유효 수면 시간 데이터 생성
df_sleep_valid = df[
    df['sleep'].notna() & (df['sleep'] >= 3) & (df['sleep'] <= 16)
].copy()

# 2. 제외된 인원 수 확인
excluded_outlier = (
    df['sleep'].notna() & ((df['sleep'] < 3) | (df['sleep'] > 16))
).sum()
print(f'• 수면 시간 이상값으로 제외된 인원: {excluded_outlier}명')

# 3. 이상값이 제외된 최종 수면 시간 기술통계 확인
print('\n=== [이상값 제외 후 수면 시간 기술통계] ===')
print(df_sleep_valid[['sleep']].describe())